# 1. Import Libraries & Data

In [4]:
import pandas as pd
import numpy as np
data_input_dir = "./data/{name}_hourly.csv"
data_cleaned_dir = "./data/{name}_hourly_CLEANED.csv"
data_filenames = [
    "AEP",
    "COMED",
    "DAYTON",
    "DEOK",
    "DOM",
    "DUQ",
    "EKPC",
    "FE",
    "NI",
    "PJM_Load",
    "PJME",
    "PJMW"
]
columns_to_drop = ["index", "level_0"]
data_frames = {}
# rename MW columns:
datetime = "Datetime"
megawatts= "MW"
for file in data_filenames:
    df = pd.read_csv(data_input_dir.format(name=file))
    #original_col = df.columns.values
    #original_col = np.delete(original_col, np.where(original_col == datetime))
    columns = [datetime, megawatts]
    df.columns = columns
    data_frames[file] = df

display(data_frames["AEP"].info())
display(data_frames["AEP"].describe().round(3))

#print(f"{data_frames.keys()} --> turn into classes?")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   MW        121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66497 entries, 0 to 66496
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Datetime  66497 non-null  object 
 1   MW        66497 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.0+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121275 entries, 0 to 121274
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121275 non-null  object 
 1   MW        121275 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None
<class 'pandas.core.frame

None

,MW
count,121273.000
mean,15499.514
std,2591.399
min,9581.000
25%,13630.000
50%,15310.000
75%,17200.000
max,25695.000


# 2. Drop Duplicates

In [204]:
def drop_duplicates(df):
    #print(f"Missing Values:\n{df.isna().sum()}\n") # missing
    duplis = df[df.duplicated(keep="first")]
    print(f"DUPLICATES: {len(duplis.index)}")
    print(f"FOUND:\n{duplis}")
    df.drop_duplicates(keep='first', inplace=True)
    df.reset_index(inplace=True, drop=True)
    if "index" in df.columns: df.drop(columns="index", inplace=True)
    isRemoved = False
    try:
        f = df.loc[duplis.index]
    except KeyError:
            isRemoved = True

    print(f"Has been removed: {isRemoved}")


print(data_frames["AEP"].info())
aep = data_frames["AEP"]
print(aep.info())
#print(f"Number of Rows w/o duplicates: {len(aep.index)}")
aep.loc[121273, "Datetime"] =  "2004-12-31 01:00:00"
aep.loc[121273, "AEP_MW"] = 13478.0
aep.loc[121274, "Datetime"] =  "2004-12-31 01:00:00"
aep.loc[121274, "AEP_MW"] = 13478.0
aep.loc[121275, "Datetime"] =  "2004-12-31 01:00:00"
aep.loc[121275, "AEP_MW"] = 13478.0




#aepRemovedDuplicates = drop_duplicates(aep)
#print(f"Number of Rows w/ duplicates: {len(aep.index)}")
drop_duplicates(aep)
#print(f"\n\nDuplicate-Free Rows:\n{aep.head()}")
aep.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None
DUPLICATES: 3
FOUND:
                   Datetime   AEP_MW
121273  2004-12-31 01:00:00  13478.0
121274  2004-12-31 01:00:00  13478.0
121275  2004-12-31 01:00:00  13478.0
Has been removed: True


,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


# 3. Drop Missing, Corrupted/Missing, Impossible Values
## Missing Values
- null-threshold: 0.3
- check rows for null-values
    - (num(null-values) a row)/num(columns) > threshold --> drop
- check columns for null-values
  - num(null-values in a column) / num(rows) > threshold --> drop

## Corrupted/Impossible Values
- valid datetime format: `YYYY-MM-dd HH:00:00`
- if a datetime-cell has an invalid format:
    - round datetime to nearest hour (minutes>30 -> round up)
    - `MW` take avg(MW) for that hour
- from the energy company label --> research operation location --> find timezone

```
<data_frame>[<column_name>].apply( lambda x: 1 if x==True else 0 )
```



## Check Rows for Null/NaN/None values, if found: drop the rows

In [205]:
def check_rows_null(df):
    mask= df.isna()
    tmp = df.loc[mask.values==True]
    print(f"\nFound {len(tmp.index)} rows w/ NaN/None/Null data:\n{tmp}\n")
    df.dropna(inplace=True)

"""
aep.loc[121273, "Datetime"] =  "2004-09-30 01:00:00"
aep.loc[121273, "AEP_MW"] = None
aep.loc[121274, "Datetime"] =  None
aep.loc[121274, "AEP_MW"] = 13478.0
aep.loc[121275, "Datetime"] =  None
aep.loc[121275, "AEP_MW"] = None
"""
check_rows_null(aep)
aep.info()


Found 0 rows w/ NaN/None/Null data:
Empty DataFrame
Columns: [Datetime, AEP_MW]
Index: []

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB


## Check Columns for amount of null values, if over the threshold: drop the column

In [206]:
def check_cols_null(df):
    old_cols = df.columns.array
    threshold = round(len(df.index) * 0.3) # threshold: 30% of the column values are null
    df.dropna(
        axis=1,
        thresh= threshold,
        inplace=True
    )
    new_cols = df.columns.array
    diff = set(old_cols) - set(new_cols)
    print(f"Removed Column: {diff}")

#df_rows = len(aep.index)
#series = pd.Series([None for i in range(df_rows)])
#aep["null_col"] = series.values
#print(aep.info())
check_cols_null(aep)
aep.info()

Removed Column: set()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB


## Check the `Datetime` column for consistency

In [207]:
from datetime import datetime, timedelta
DATE_REGEX = r"^\d{4}-\d{2}-\d{2} \d{2}:00:00$" # YYYY-MM-dd HH:00:00

#col = "Datetime"
aep.loc[0, datetime] = pd.NaT
aep.loc[0, megawatts] = 1000.0
aep.loc[121273, datetime] = pd.NaT
aep.loc[121273, megawatts] = 1000.0
aep.loc[121274, datetime] = pd.NaT
aep.loc[121274, megawatts] = 1000.0
aep.loc[121275, datetime] = pd.Timestamp("2020-05-05 12:35:55")
aep.loc[121275, megawatts] = 1000.0

def format_dates(df):
    DATE_REGEX = r"^\d{4}-\d{2}-\d{2} \d{2}:00:00$" # YYYY-MM-dd HH:00:00
    # attempts to transform all dates into dates, if it fails it replaces it with NaT (Not a Time)
    parsed = pd.to_datetime(df[datetime], errors="coerce")
    # mask of NaT values
    mask = parsed.isna()
    # rows w/ NaT as dates
    not_a_time = df[mask.values == True]

    # successfully converted rows
    true_dates = df[mask.values==False]
    s = true_dates[datetime].astype("string")
    bad = ~s.str.match(DATE_REGEX, na=False)
    not_full = true_dates[bad.values == True]
    tmp =not_full[datetime].astype(str).str.split(":")


    g_i = list(set(true_dates.index) - set(tmp.index))
    good_dates = true_dates.loc[g_i]
    good_dates[datetime] = pd.to_datetime(good_dates[datetime])

    # Not a full hour
    for i in tmp.index:
        truncated_date = pd.Timestamp(f"{tmp.loc[i][0]}:00:00")

        f = good_dates[good_dates[datetime].dt.hour == truncated_date.hour]
        mean = good_dates.loc[f.index, megawatts].mean()

        tmp.loc[i] = truncated_date
        data = {
            datetime: truncated_date,
            megawatts: mean
        }
        df.loc[i] = data


    # NaT
    for i in not_a_time.index:
        below = -1
        try:
            below = i - 1
            aep.loc[below]
        except KeyError:
            try:
                below = i+1
                df.loc[below]
            except KeyError:
                raise Exception(f"Failed to get above/below: {i}")
        finally:
            data = {}
            isValidDate = False
            if below in range(0,len(df.index)):
                belowDate = pd.to_datetime(df.loc[below, datetime])
                newDate = belowDate + timedelta(hours=1)
                print(newDate)
                print(newDate in df[datetime].values)
                if newDate not in df[datetime].values:
                    data[datetime] = newDate
                    isValidDate = True
                else:
                    newDate = pd.to_datetime(df.loc[below+2, datetime]) + timedelta(hours=1)
                    if newDate not in df[datetime].values:
                        data[datetime] = newDate
                        isValidDate = True
                if isValidDate:
                    filtered = parsed[parsed.dt.hour == newDate.hour]
                    mean = df.loc[filtered.index, megawatts].mean()
                    data[megawatts]= mean
                    df.loc[i] = data
                    print(f"Added:\n{df.loc[i]}")
                else:
                    print(f"dropped:\n{df.loc[i]}")
                    df.drop(index=i, inplace=True)
            df[datetime] = pd.to_datetime(df[datetime], errors="coerce")
            df[df[datetime].isna().values ==True]

2004-12-31 03:00:00
False
Added:
Datetime    2004-12-31 03:00:00
AEP_MW             13184.049008
Name: 0, dtype: object
2018-01-02 01:00:00
False
Added:
Datetime    2018-01-02 01:00:00
AEP_MW             13891.560261
Name: 121273, dtype: object
2018-01-02 02:00:00
False
Added:
Datetime    2018-01-02 02:00:00
AEP_MW             13432.062995
Name: 121274, dtype: object


,Datetime,AEP_MW
